# Exercise 25 - Parking cleanup

In this exercise, we will identify missing values, one of the most common problems you will encounter. We’ll see how often values are missing and what effect they may have. Note that for this exercise, we’re going to assume that a parking ticket that is missing data may be dismissed; don’t blame me if this defense doesn’t work when appealing any tickets you get in New York.

I want you to do the following:

1. Create a data frame from the file ```nyc-parking-violations-2020.csv```. We are only interested in a handful of the columns: ```Plate ID```, ```Registration State```, ```Vehicle Make```, ```Vehicle Color```, ```Violation Time```, ```Street Name```.

How many rows are in the data frame when it is read into memory?

In [27]:
import pandas as pd

parking_data = pd.read_csv(
    "./nyc-parking-violations-2020.csv",
    header=0,
    usecols=["Plate ID", "Registration State", "Vehicle Make", "Vehicle Color", "Violation Time", "Street Name"]
)

In [28]:
n_rows = parking_data.shape[0] # number of rows

print(f"Number of rows: {n_rows:,}")

Number of rows: 12,495,734


2. Remove rows with any missing data (i.e., a ```NaN``` value). How many rows remain after doing this pruning? If each parking ticket brings $100 into the city, and missing data means the ticket can be successfully contested, how much money may New York City lose due to such missing data?

In [29]:
n_rows_clean = parking_data.dropna().shape[0] # number of rows after removing nan values
deleted_rows = n_rows - n_rows_clean
ticket_fare = 100 # ticket fare is $100

print(f"Number of deleted rows: {deleted_rows:,}\n"
      f"Money that can be lost due to missing data: ${deleted_rows * ticket_fare:,.2f}"
      )



Number of deleted rows: 447,359
Money that can be lost due to missing data: $44,735,900.00


3. Let’s instead assume that a ticket can only be dismissed if the license plate, state, car make, and/or street name are missing. Remove rows that are missing one or more of these. How many rows remain? Assuming $100/ticket, how much money would the city lose as a result of this missing data?

In [30]:
# number of rows with nan values in specific columns
n_nan_rows = parking_data.loc[
    parking_data["Plate ID"].isna() | # select rows where the plate id is missing
    parking_data["Registration State"].isna() | # select rows where the registration state is missing
    parking_data["Vehicle Make"].isna() | # select rows where vehicle make is missing
    parking_data["Street Name"].isna() # select rows where street name is missing
].shape[0]

clean_rows = n_rows - n_nan_rows

print(f"Number of clean rows: {clean_rows:,}\n"
      f"Money that can be lost due to missing data: ${n_nan_rows * ticket_fare:,.2f}"
      )


Number of clean rows: 12,431,949
Money that can be lost due to missing data: $6,378,500.00


4. Now let’s assume that tickets can be dismissed if the license plate, state, and/or street name are missing—that is, the same as the previous question, but without requiring the make of car. Remove rows that are missing one or more of these. How many rows remain? Assuming $100/ticket, how much money would the city lose as a result of this missing data?

In [31]:
# number of rows with nan values in specific columns
n_nan_rows = parking_data.loc[
    parking_data["Plate ID"].isna() | # select rows where the plate id is missing
    parking_data["Registration State"].isna() | # select rows where the registration state is missing
    parking_data["Street Name"].isna() # select rows where street name is missing
].shape[0]

clean_rows = n_rows - n_nan_rows

print(f"Number of clean rows: {clean_rows:,}\n"
      f"Money that can be lost due to missing data: ${n_nan_rows * ticket_fare:,.2f}"
      )

Number of clean rows: 12,494,116
Money that can be lost due to missing data: $161,800.00


## Beyond the exercise

* So far, you have specified which columns must be all non-null. But sometimes it’s OK for some columns to have null values, as long as it’s not too many. How many rows would you eliminate if you required at least three non-null values from the four columns ```Plate ID```, ```Registration State```, ```Vehicle Make```, and ```Street Name```?

In [32]:
n_rows_clean = parking_data.dropna(
    subset=[
      "Plate ID",
      "Registration State",
      "Vehicle Make",
      "Street Name",
  ],
  thresh=3).shape[0]

deleted_rows = n_rows - n_rows_clean

print(f"Number of deleted rows: {deleted_rows:,}")

Number of deleted rows: 253


* Which of the columns you’ve imported has the greatest number of ```NaN``` values? Is this a problem?

In [33]:
colum_max_nan = parking_data.isna().sum().sort_values(ascending=False)
colum_max_nan

Vehicle Color         391982
Vehicle Make           62420
Street Name             1417
Violation Time           278
Plate ID                 202
Registration State         0
dtype: int64

* Null data is bad, but there is plenty of bad non-null data, too. For example, many cars with ```BLANKPLATE``` as a plate ID were ticketed. Turn these into ```NaN``` values, and rerun the previous query.

In [34]:
parking_data2 = parking_data.replace({"Plate ID": "BLANKPLATE"}, pd.NA) # change BLANKPLATE to nan

colum_max_nan = parking_data2.isna().sum().sort_values(ascending=False)
colum_max_nan

Vehicle Color         391982
Vehicle Make           62420
Plate ID                9084
Street Name             1417
Violation Time           278
Registration State         0
dtype: int64

Before changing ```BLANKPLATE``` to ```NAN``` there were 202 Plate IDs with ```NAN``` values, now there are 9084!!